In [0]:
%run ../functions/functions

In [0]:
# ==============================
# Lista dos datasets
# ==============================

datasets = [
    "EXP_2021", 
    "EXP_2021_MUN",
    "EXP_2022", 
    "EXP_2022_MUN",
    "IMP_2021", 
    "IMP_2021_MUN",
    "IMP_2022", 
    "IMP_2022_MUN",
    "ISIC_CUCI",
    "NBM",
    "NBM_NCM", 
    "NCM",
    "NCM_CGCE",
    "NCM_CUCI",
    "NCM_FAT_AGREG",
    "NCM_ISIC",
    "NCM_PPE",
    "NCM_PPI",
    "NCM_SH",
    "NCM_UNIDADE",
    "PAIS",
    "PAIS_BLOCO",
    "UF",
    "UF_MUN",
    "URF",
    "VIA"
]

In [0]:
#funcao criada para popular os datasets
def load_all_datasets(lista):
    dfs = {}
    for dataset in lista:
        dfs[dataset] = read_bronze(dataset)
    return dfs

In [0]:
dfs = load_all_datasets(datasets)

In [0]:
#print dos Schemas da bronze
for nome, df in dfs.items():
    print(nome)
    df.printSchema()
    print("-------------------------------------------------------------")

In [0]:
#imports necessarios para o tratamento das colunas
from pyspark.sql.types import IntegerType, DoubleType, LongType
from functools import reduce
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from functools import reduce
from delta.tables import DeltaTable

In [0]:

def process_silver_layer(dfs_dict, cast_config, business_keys, sk_name):
    """
    Função genérica para unir múltiplos DataFrames e aplicar transformações da camada Silver.
    
    Args:
        dfs_dict (dict): Dicionário de DataFrames { "nome": df }.
        cast_config (dict): Dicionário mapeando coluna para seu tipo Spark (ex: {"CO_ANO": "int"}).
        business_keys (list): Lista de colunas para compor a Surrogate Key.
        sk_name (str): Nome da coluna de Surrogate Key a ser criada.
        
    Returns:
        DataFrame: DataFrame único transformado.
    """
    
    # 1. União dos DataFrames 
    # Usamos reduce para unir todos os DataFrames presentes no dicionário
    df_unified = reduce(DataFrame.unionAll, dfs_dict.values())
    
    # 2. Aplicação dinâmica de Casts
    # Percorre o dicionário de configuração de tipos
    for col_name, col_type in cast_config.items():
        if col_name in df_unified.columns:
            df_unified = df_unified.withColumn(col_name, F.col(col_name).cast(col_type))
    
    # 3. Criação da Surrogate Key (SK) via MD5
    # Garante que nulos não quebrem a chave e usa um separador para evitar colisões
    df_transformed = df_unified.withColumn(
        sk_name,
        F.md5(F.concat_ws("|", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in business_keys]))
    )
    
    # 4. Organização final: SK na primeira posição
    cols = [sk_name] + [c for c in df_transformed.columns if c != sk_name]
    
    return df_transformed.select(cols)


In [0]:
# 1. Configurações específicas para o dataset de Exportação (EXP)
exp_cast_config = {
    "CO_ANO": "int",
    "CO_MES": "int",
    "CO_NCM": "long",
    "CO_UNID": "int",
    "CO_PAIS": "int",
    "CO_VIA": "int",
    "CO_URF": "int",
    "QT_ESTAT": "double",
    "KG_LIQUIDO": "double",
    "VL_FOB": "double"
}

exp_business_keys = [
    "CO_ANO", "CO_MES", "CO_NCM", "CO_UNID", 
    "CO_PAIS", "SG_UF_NCM", "CO_VIA", "CO_URF"
]


In [0]:
# 2. Execução do pipeline
datasets = ["EXP_2021", "EXP_2022"]
dfs = load_all_datasets(datasets)

df_silver_exp = process_silver_layer(
    dfs_dict=dfs, 
    cast_config=exp_cast_config, 
    business_keys=exp_business_keys, 
    sk_name="SK_EXPORTACAO"
)

df_silver_exp.display()

In [0]:
df_silver_exp.display()

In [0]:
#validacao do union
dfs["EXP_2021"].count() + dfs["EXP_2022"].count() #- df_silver_exp.count()

In [0]:

def save_silver_incremental(df, table_name, primary_keys):
    """
    Salva os dados na camada Silver usando o formato Delta com lógica de MERGE (Upsert).
    Isso evita duplicatas e é muito mais performático que joins manuais em Parquet.
    """
    print(f"\nIniciando Processamento Silver: {table_name}")
    
    # Caminho da tabela (Azure Data Lake Storage Gen2)
    silver_path = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/{table_name}/"
    
    # 1. Limpeza e Deduplicação Interna (Batch Atual)
    # Assumindo que a função clean_and_fill já existe no seu ambiente
    string_cols = [c for c, t in df.dtypes if t == "string"]
    df = clean_and_fill(df, string_cols)
    
    # Remove duplicatas que possam vir no mesmo lote de processamento
    df = df.dropDuplicates(primary_keys)
    
    # 2. Operação Delta Lake
    try:
        # Verifica se o caminho já contém uma tabela Delta válida
        if DeltaTable.isDeltaTable(spark, silver_path):
            print(f"Tabela Delta encontrada em {table_name}. Realizando MERGE...")
            
            delta_table = DeltaTable.forPath(spark, silver_path)
            
            # Construção dinâmica da condição de junção para o Merge
            # Ex: "target.SK_ID = updates.SK_ID AND target.CO_ANO = updates.CO_ANO"
            merge_condition = " AND ".join([f"target.{k} = updates.{k}" for k in primary_keys])
            
            delta_table.alias("target").merge(
                df.alias("updates"),
                merge_condition
            ).whenNotMatchedInsertAll().execute()
            
            print(f"Merge finalizado com sucesso para {table_name}")
        else:
            raise Exception("Caminho existe mas não é uma tabela Delta")

    except Exception as e:
        # Se a tabela não existir ou ocorrer erro na validação Delta, fazemos o overwrite inicial
        print(f"Criando nova tabela Delta (Primeira Carga ou Erro): {table_name}")
        
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(silver_path)
            
        print(f"Tabela Delta criada com sucesso em: {silver_path}")


In [0]:
# Salvar de forma incremental na Silver usando Delta
# A primary_key aqui deve ser a sua Surrogate Key criada no passo anterior
save_silver_incremental(
    df=df_silver_exp, 
    table_name="EXP_CONSOLIDADA", 
    primary_keys=["SK_EXPORTACAO"]
)

In [0]:
# Definindo o caminho
silver_path = "abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/EXP_CONSOLIDADA/"

# Lendo os dados
df_silver = spark.read.format("delta").load(silver_path)

# Verificando o resultado
#df_silver.count()

In [0]:
df_silver.count()
2976163